# ⚖️ LAB 1 — Naive RAG para Documentos Jurídicos
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Construir um sistema RAG completo do zero, célula por célula, com documentos jurídicos reais.

**Duração estimada:** 120 minutos

---
### 🗺️ Diagrama Completo do Pipeline

```
╔══════════════════════════════════════════════════════════════════╗
║                     NAIVE RAG PIPELINE                           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  FASE 1 — INDEXAÇÃO (offline, feita uma vez)                    ║
║                                                                  ║
║  Docs Jurídicos                                                  ║
║      │                                                           ║
║      ▼                                                           ║
║  [RecursiveCharacterTextSplitter]                                ║
║      │  chunk_size=800, overlap=150                             ║
║      ▼                                                           ║
║  Chunks (LangChain Documents)                                    ║
║      │                                                           ║
║      ▼                                                           ║
║  [OllamaEmbeddings] (BGE-M3 servido pelo Ollama da Aula 1)       ║
║      │  ollama pull bge-m3                                       ║
║      ▼                                                           ║
║  Vetores (float32, dim=1024)                                     ║
║      │                                                           ║
║      ▼                                                           ║
║  [FAISS VectorStore]  ◄── Índice salvo em disco                 ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  FASE 2 — RETRIEVAL + GERAÇÃO (online, por query)               ║
║                                                                  ║
║  Pergunta do Usuário                                             ║
║      │                                                           ║
║      ▼                                                           ║
║  [OllamaEmbeddings(bge-m3)]  ← mesmo modelo da indexação        ║
║      │                                                           ║
║      ▼                                                           ║
║  [FAISS.similarity_search]                                       ║
║      │  top_k=3 chunks mais similares                           ║
║      ▼                                                           ║
║  Contexto Recuperado                                             ║
║      │                                                           ║
║      ├─ + Pergunta Original                                      ║
║      ▼                                                           ║
║  [Prompt Template]                                               ║
║      │                                                           ║
║      ▼                                                           ║
║  [LLM - Ollama (llama3.2:3b) em http://localhost:11434]                 ║
║      │                                                           ║
║      ▼                                                           ║
║  Resposta Fundamentada + Citações                               ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

> **ABNT:** LEWIS, P. et al. *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS, 2020.

## 🔧 Célula 1 — Instalação

Instalamos todas as dependências necessárias. Execute apenas uma vez por sessão.

In [ ]:
# ============================================================
# CÉLULA 1/12 — INSTALAÇÃO DE DEPENDÊNCIAS
# Compatível com o venv_rag da Aula 1 (Ollama + OpenSearch + LangFuse)
# ============================================================

# Core RAG stack
!pip install -q langchain langchain-community langchain-text-splitters

# Vector store
!pip install -q faiss-cpu

# Embeddings + LLM via Ollama (servidor local da Aula 1)
!pip install -q langchain-ollama

# Fallback HuggingFace (caso não consiga puxar 'bge-m3' no Ollama)
!pip install -q sentence-transformers

# Alternativa portátil: ChatOpenAI apontando para o endpoint /v1 do Ollama
!pip install -q langchain-openai openai

# Visualizações
!pip install -q pandas matplotlib

# Docling para ingestão
!pip install -q docling

print('✅ Todas as dependências instaladas!')
print()
print('📦 Stack do Exemplo (mesma infra da Aula 1):')
print('   LangChain      → Orquestração do pipeline RAG')
print('   FAISS          → Vector Store local (com fallback)')
print('   Ollama         → Servidor único para LLM e Embeddings (porta 11434)')
print('   bge-m3 @ Ollama → Embeddings BGE-M3 multilíngue (dim=1024)')
print('   llama3.2:3b @ Ollama → LLM padrão para geração')
print('   Docling        → Ingestão de PDFs')


## 📚 Célula 2 — Corpus Jurídico

Criamos um corpus realista com 5 documentos de diferentes tipos:
- Acórdão (habeas corpus)
- Legislação (Código Penal)
- Relatório de inteligência
- Parecer do Ministério Público
- Súmula do STJ

In [ ]:
# ============================================================
# CÉLULA 2/12 — CORPUS JURÍDICO
# Dois caminhos:
#   (A) CORPUS_JURIDICO inline (default, rápido — exemplo didático)
#   (B) Ingestão via Docling dos PDFs reais do dataset da Aula 2
#       (Manual_DPCA_atualizado.pdf + Laudo.pdf)
#
# Mude USE_DOCLING_REAL=True para usar os PDFs reais.
# ============================================================
from pathlib import Path
from langchain.schema import Document

USE_DOCLING_REAL = False    # True = carrega Manual_DPCA + Laudo via Docling

DATASET_DIR = Path("../datasets").resolve()
PDF_MANUAL  = DATASET_DIR / "Manual_DPCA_atualizado.pdf"
PDF_LAUDO   = DATASET_DIR / "Laudo.pdf"

if USE_DOCLING_REAL and PDF_MANUAL.exists() and PDF_LAUDO.exists():
    # ────────────────────────────────────────────────────────
    # OPÇÃO B — Docling + PDFs reais
    # ────────────────────────────────────────────────────────
    print("🟢 Modo: DOCLING + PDFs REAIS")
    from docling.document_converter import DocumentConverter
    from docling.datamodel.pipeline_options import PipelineOptions
    conv_rapido = DocumentConverter(pipeline_options=PipelineOptions(do_ocr=False, do_table_structure=True))
    conv_ocr    = DocumentConverter(pipeline_options=PipelineOptions(do_ocr=True,  do_table_structure=True))
    print("⏳ Convertendo Manual_DPCA_atualizado.pdf (sem OCR)...")
    md_manual = conv_rapido.convert(str(PDF_MANUAL)).document.export_to_markdown()
    print(f"   ✓ {len(md_manual)} chars extraídos")
    print("⏳ Convertendo Laudo.pdf (com OCR — pode demorar)...")
    md_laudo = conv_ocr.convert(str(PDF_LAUDO)).document.export_to_markdown()
    print(f"   ✓ {len(md_laudo)} chars extraídos via OCR")

    CORPUS_JURIDICO = [
        Document(page_content=md_manual, metadata={"fonte": "Manual_DPCA", "tipo": "manual_operacional", "origem": "Docling"}),
        Document(page_content=md_laudo,  metadata={"fonte": "Laudo_Pericial", "tipo": "laudo", "origem": "Docling+OCR"}),
    ]
else:
    # ────────────────────────────────────────────────────────
    # OPÇÃO A — CORPUS inline (default, rápido)
    # ────────────────────────────────────────────────────────
    if USE_DOCLING_REAL:
        print(f"⚠️  USE_DOCLING_REAL=True mas PDFs não encontrados em {DATASET_DIR}")
        print("   Caindo para o CORPUS_JURIDICO inline.\n")
    print("🟡 Modo: CORPUS_JURIDICO inline (rápido)\n")

    CORPUS_JURIDICO = [
    Document(
        page_content="""
        ACÓRDÃO - HABEAS CORPUS Nº 2025.001234-SP
        Tribunal de Justiça do Estado de São Paulo
        7ª Câmara Criminal

        EMENTA: HABEAS CORPUS. TRÁFICO DE DROGAS. PRISÃO PREVENTIVA.
        A prisão preventiva é medida cautelar de ultima ratio, devendo ser decretada somente
        quando as demais medidas cautelares do artigo 319 do CPP se mostrarem insuficientes.
        A fundamentação exclusivamente na gravidade abstrata do delito não constitui fundamento
        idôneo para a custódia cautelar. Precedentes do STJ: HC 123.456/SP.
        ORDEM CONCEDIDA para revogar a prisão preventiva.

        FUNDAMENTAÇÃO: O réu FULANO foi preso preventivamente há 145 dias por tráfico de
        drogas (Art. 33 da Lei 11.343/2006). A instrução criminal não foi concluída.
        O juízo fundamentou a preventiva apenas na gravidade abstrata do crime e na
        quantidade de droga apreendida (50g de cocaína), sem indicar risco concreto.
        O paciente é primário, tem residência fixa no Bairro Jardim das Flores,
        São Paulo/SP, e renda lícita como motorista de aplicativo.
        """,
        metadata={
            "fonte": "TJSP",
            "tipo": "acórdão",
            "numero": "HC-2025.001234-SP",
            "assunto": "habeas corpus - prisão preventiva",
            "data": "2025-03-15"
        }
    ),
    Document(
        page_content="""
        CÓDIGO PENAL BRASILEIRO — ARTIGOS RELEVANTES

        Art. 33. Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, vender,
        expor à venda, oferecer, ter em depósito, transportar, trazer consigo, guardar,
        prescrever, ministrar, entregar a consumo ou fornecer drogas, ainda que gratuitamente,
        sem autorização ou em desacordo com determinação legal ou regulamentar:
        Pena: reclusão de 5 (cinco) a 15 (quinze) anos e pagamento de 500 (quinhentos)
        a 1.500 (mil e quinhentos) dias-multa.

        Art. 28. Quem adquirir, guardar, tiver em depósito, transportar ou trouxer consigo,
        para consumo pessoal, drogas sem autorização ou em desacordo com determinação
        legal ou regulamentar será submetido às seguintes penas:
        I - advertência sobre os efeitos das drogas;
        II - prestação de serviços à comunidade;
        III - medida educativa de comparecimento a programa ou curso educativo.

        Art. 312 CPP. A prisão preventiva poderá ser decretada como garantia da ordem pública,
        da ordem econômica, por conveniência da instrução criminal, ou para assegurar a
        aplicação da lei penal, quando houver prova da existência do crime e indício suficiente
        de autoria e perigo gerado pelo estado de liberdade do imputado.
        """,
        metadata={
            "fonte": "Legislação Federal",
            "tipo": "legislação",
            "numero": "Lei 11.343/2006",
            "assunto": "tráfico de drogas - uso pessoal - código penal",
            "data": "2006-08-23"
        }
    ),
    Document(
        page_content="""
        RELATÓRIO DE INTELIGÊNCIA POLICIAL — Nº 0042/2025
        Delegacia de Repressão ao Narcotráfico — DENARC
        Período: 1º Trimestre 2025

        SUMÁRIO EXECUTIVO:
        Foram registradas 113 ocorrências de tráfico de entorpecentes no 1º trimestre de 2025,
        representando aumento de 11,8% em relação ao mesmo período de 2024.
        Os pontos críticos identificados são: Bairro Jardim das Flores, Vila Operária e
        Conjunto Habitacional Santos Dumont.

        ANÁLISE OPERACIONAL:
        A principal organização criminosa identificada opera com células de 4 a 6 membros.
        Utiliza aplicativos de mensagens criptografadas para comunicação.
        Apreensões no trimestre: 2,3kg cocaína, 1,8kg crack, 15kg maconha.
        Foram presos 34 indivíduos, sendo 8 considerados lideranças.

        RECOMENDAÇÕES:
        1. Intensificar operações nos 3 pontos críticos identificados;
        2. Solicitar apoio do GAECO para investigação de lideranças;
        3. Articular com Ministério Público para denúncias de associação.
        """,
        metadata={
            "fonte": "DENARC-SP",
            "tipo": "relatório_inteligência",
            "numero": "RI-0042/2025",
            "assunto": "tráfico drogas - estatísticas - operações",
            "data": "2025-04-01"
        }
    ),
    Document(
        page_content="""
        PARECER DO MINISTÉRIO PÚBLICO — Processo 0001234-56.2024.8.26.0001

        Excelentíssimo Senhor Desembargador Relator,

        O Ministério Público opina pelo NÃO CONHECIMENTO do presente writ e,
        subsidiariamente, pela DENEGAÇÃO DA ORDEM, pelos seguintes fundamentos:

        I) DA INADEQUAÇÃO DA VIA ELEITA:
        O habeas corpus não é instrumento adequado para o reexame de provas.
        O constrangimento ilegal não é manifesto, exigindo análise aprofundada dos autos.

        II) DA MANUTENÇÃO DA PRISÃO PREVENTIVA:
        A quantidade de droga apreendida — 50g de cocaína — demonstra que o acusado
        não se trata de simples usuário, mas de traficante com capacidade de abastecimento.
        A periculosidade concreta do acusado está demonstrada pelas circunstâncias do flagrante.
        A liberdade do acusado representa risco à ordem pública e à instrução criminal,
        pois há testemunhas a serem ouvidas que podem sofrer intimidação.

        São Paulo, 10 de março de 2025.
        Dr. PROMOTOR DE JUSTIÇA
        4ª Promotoria de Justiça Criminal — SP
        """,
        metadata={
            "fonte": "MPSP",
            "tipo": "parecer",
            "numero": "Processo-0001234-56.2024",
            "assunto": "habeas corpus - parecer ministerial",
            "data": "2025-03-10"
        }
    ),
    Document(
        page_content="""
        SÚMULAS E JURISPRUDÊNCIA — STJ/STF

        SÚMULA 492/STJ: O ato infracional análogo ao tráfico de drogas, por si só,
        não conduz obrigatoriamente à imposição de medida socioeducativa de internação
        do adolescente.

        SÚMULA 70/STJ: É inadmissível a prisão civil do depositário infiel.

        ORIENTAÇÃO JURISPRUDENCIAL STJ — PRISÃO PREVENTIVA:
        A jurisprudência consolidada do Superior Tribunal de Justiça firmou o entendimento
        de que a gravidade abstrata do delito, por si só, não constitui fundamentação
        idônea para a decretação ou manutenção da prisão preventiva.
        É necessária a demonstração concreta do risco que o estado de liberdade do
        acusado representa para a ordem pública ou para a instrução criminal.

        TESE FIXADA NO TEMA 366/STF:
        A quantidade de droga não é o único critério para a distinção entre tráfico
        e consumo pessoal, devendo ser consideradas as circunstâncias do flagrante.
        """,
        metadata={
            "fonte": "STJ/STF",
            "tipo": "jurisprudência",
            "numero": "Súmulas diversas",
            "assunto": "súmulas - teses - prisão preventiva",
            "data": "2024-01-01"
        }
    )
]

print(f"\n✓ Corpus carregado: {len(CORPUS_JURIDICO)} Documents")
for i, d in enumerate(CORPUS_JURIDICO):
    print(f"   [{i+1}] {d.metadata.get('fonte', 'N/A'):20s} | {len(d.page_content)} chars")


## ✂️ Célula 3 — Chunking dos Documentos

Aplicamos o `RecursiveCharacterTextSplitter` com parâmetros calibrados para textos jurídicos.

In [ ]:
# ============================================================
# CÉLULA 3/12 — CHUNKING
# Estratégia: Recursive Character (melhor para textos jurídicos)
# ============================================================
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Parâmetros calibrados para documentos jurídicos brasileiros
# chunk_size=800: captura ~1-2 parágrafos completos
# chunk_overlap=150: garante continuidade entre chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", "; ", ",", " ", ""],
)

# split_documents() preserva os metadados de cada Document original
# Isso é diferente de split_text() que retorna strings puras
chunks = text_splitter.split_documents(CORPUS_JURIDICO)

print(f"✂️  RESULTADO DO CHUNKING")
print(f"{'='*60}")
print(f"Documentos originais: {len(CORPUS_JURIDICO)}")
print(f"Chunks gerados:       {len(chunks)}")
print(f"Média chars/chunk:    {sum(len(c.page_content) for c in chunks)//len(chunks)}")
print()

# Mostra distribuição por tipo de documento
from collections import Counter
tipos = Counter(c.metadata.get('tipo', 'N/A') for c in chunks)
print("Distribuição por tipo:")
for tipo, count in sorted(tipos.items()):
    print(f"  {tipo:25} → {count} chunks")

print()
print("📌 Verificação: metadados preservados?")
print(f"  Chunk 1 metadados: {chunks[0].metadata}")
print(f"  Chunk 1 conteúdo:  {chunks[0].page_content[:150].strip()!r}")

## 🔢 Célula 4 — Geração de Embeddings

Geramos vetores numéricos para cada chunk. Usamos um modelo **multilíngue local** (sem API key).

In [ ]:
# ============================================================
# CÉLULA 4/12 — EMBEDDINGS
# Padrão da Aula 2: BGE-M3 servido pelo Ollama da Aula 1
# Fallback: HuggingFace BAAI/bge-m3 (download local) se Ollama indisponível
# Dimensão: 1024 | Idiomas: 100+ (incluindo PT-BR)
# ============================================================
import os, requests
from dotenv import load_dotenv
load_dotenv()
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

print('⏳ Inicializando embeddings...')
embeddings_model = None

try:
    from langchain_ollama import OllamaEmbeddings
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
    embeddings_model = OllamaEmbeddings(
        model=OLLAMA_EMBED_MODEL,
        base_url=OLLAMA_BASE_URL,
    )
    teste = embeddings_model.embed_query('teste')
    EMBEDDING_MODEL_NAME = f'ollama:{OLLAMA_EMBED_MODEL}'
    print(f'✅ Embeddings Ollama OK | modelo={OLLAMA_EMBED_MODEL} | dim={len(teste)}')
except Exception as e:
    print(f'⚠️  Ollama indisponível ({e}). Caindo para HuggingFace BAAI/bge-m3.')
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings_model = HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True, 'batch_size': 32},
    )
    EMBEDDING_MODEL_NAME = 'huggingface:BAAI/bge-m3'
    print(f'✅ HuggingFaceEmbeddings carregado | modelo={EMBEDDING_MODEL_NAME}')

# ATENÇÃO: Mantenha EXATAMENTE o mesmo modelo aqui e na query.
# Trocar o modelo de embedding corrompe o índice.

# Teste rápido para verificar o modelo
teste_embedding = embeddings_model.embed_query('prisão preventiva tráfico drogas')
print(f'\n📐 Dimensão do embedding: {len(teste_embedding)}')
print(f'   Primeiros 5 valores: {[round(v, 4) for v in teste_embedding[:5]]}')
print(f'   Norma do vetor: {sum(v**2 for v in teste_embedding)**0.5:.4f} (≈1.0 quando normalizado)')


## 🗂️ Célula 5 — Criando o Índice FAISS

FAISS (Facebook AI Similarity Search) é uma biblioteca de busca por vizinhos mais próximos extremamente eficiente. Para nosso corpus de 5 documentos, o índice é construído em segundos.

In [ ]:
# ============================================================
# CÉLULA 5/12 — CRIANDO O ÍNDICE FAISS
# ============================================================
from langchain.vectorstores import FAISS
import time

print("⏳ Gerando embeddings e construindo índice FAISS...")
print(f"   Chunks a indexar: {len(chunks)}")
print(f"   Modelo: {EMBEDDING_MODEL}")

inicio = time.time()

# from_documents(): gera embeddings de todos os chunks E cria o índice
# Internamente:
#   1. Chama embeddings_model.embed_documents([chunk.page_content for chunk in chunks])
#   2. Cria índice FAISS IndexFlatL2 (busca exata por distância euclidiana)
#   3. Armazena vetores + metadados
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
)

tempo = time.time() - inicio
print(f"✅ Índice FAISS criado em {tempo:.2f}s")
print()
print(f"📊 Estatísticas do índice:")
print(f"   Vetores indexados: {vectorstore.index.ntotal}")
print(f"   Dimensão: {vectorstore.index.d}")

# Salvando índice em disco (para reutilização sem re-indexar)
FAISS_PATH = "/tmp/faiss_juridico"
vectorstore.save_local(FAISS_PATH)
print(f"   Índice salvo em: {FAISS_PATH}")
print()
print("💡 Em produção: salve o índice no Google Drive para não re-indexar")

## 🔍 Célula 6 — Teste de Retrieval

Antes de conectar o LLM, testamos o retrieval isoladamente. Isso ajuda a diagnosticar problemas.

In [ ]:
# ============================================================
# CÉLULA 6/12 — TESTE DE RETRIEVAL (sem LLM ainda)
# ============================================================

def testar_retrieval(query: str, k: int = 3) -> list:
    """
    Testa o retrieval sem LLM.
    Útil para diagnosticar se o problema é no retrieval ou na geração.

    Args:
        query: Pergunta do usuário
        k: Número de chunks a recuperar

    Returns:
        Lista de (Document, score) ordenados por relevância
    """
    # similarity_search_with_score retorna (doc, score)
    # Score é distância L2: menor = mais similar
    resultados = vectorstore.similarity_search_with_score(query, k=k)

    print(f"\n🔍 Query: '{query}'")
    print(f"   Top-{k} resultados:")
    print()

    for i, (doc, score) in enumerate(resultados, 1):
        # Convertemos L2 para similaridade (0-1) para melhor interpretação
        similaridade = 1 / (1 + score)
        print(f"  [{i}] Score L2: {score:.4f} | Similaridade: {similaridade:.4f}")
        print(f"      Fonte: {doc.metadata.get('fonte')} | Tipo: {doc.metadata.get('tipo')}")
        print(f"      Conteúdo: {doc.page_content[:200].strip()!r}")
        print()

    return resultados


# Teste 1: Query diretamente relacionada a um documento
r1 = testar_retrieval("Quais são os requisitos para a prisão preventiva?", k=3)

# Teste 2: Query sobre estatísticas
r2 = testar_retrieval("Quantas ocorrências de tráfico foram registradas em 2025?", k=2)

# Teste 3: Query sobre legislação
r3 = testar_retrieval("Qual é a pena para tráfico de drogas?", k=2)

## 🤖 Célula 7 — Configuração do LLM (Ollama Server da Aula 1)

Usamos o **Ollama** como servidor LLM local (já provisionado na Aula 1). Ele oferece **dois endpoints simultâneos**:
uma API nativa (`/api/chat`, `/api/embeddings`) e um endpoint **OpenAI-compatível** em `/v1`.
Isso permite usar `langchain-ollama` (caminho nativo) **ou** `ChatOpenAI` apontando para `http://localhost:11434/v1`
(caminho portátil — útil para reaproveitar código escrito para OpenAI ou vLLM).

```
ARQUITETURA OLLAMA (LOCAL — Aula 1)

  ┌────────────────────────────────────────────────────────┐
  │             Máquina do aluno (qualquer SO)              │
  │                                                        │
  │  ┌──────────────────┐    ┌────────────────────────┐    │
  │  │  Notebook Python │    │  Ollama daemon         │    │
  │  │  (RAG pipeline)  │───▶│  porta 11434           │    │
  │  │                  │    │   /api/chat            │    │
  │  │  ChatOllama OU   │    │   /api/embeddings      │    │
  │  │  ChatOpenAI(     │    │   /v1/chat/completions │    │
  │  │   base_url=      │    │                        │    │
  │  │  'localhost:     │    │  Modelos servidos:      │    │
  │  │   11434[/v1]')   │    │   • llama3.2:3b         │    │
  │  └──────────────────┘    │   • bge-m3              │    │
  │                          └────────────────────────┘    │
  └────────────────────────────────────────────────────────┘
```

> **Por que Ollama nesta disciplina?** É a infraestrutura provisionada na **Aula 1** — funciona em Windows/macOS/Linux
> sem CUDA, com instalação em minutos. Para produção com alto QPS sobre Linux+GPU NVIDIA, o vLLM continua sendo
> uma boa escolha; a migração é trivial porque o Ollama também expõe a API OpenAI-compatível (`/v1`).


In [ ]:
# ============================================================
# CÉLULA 7/12 — CONFIGURAÇÃO DO LLM
# OPÇÃO A: Ollama nativo (RECOMENDADO — infra da Aula 1)
# OPÇÃO B: ChatOpenAI apontando para o endpoint /v1 do Ollama (portátil)
# OPÇÃO C: HuggingFace Pipeline local (offline, sem servidor)
# ============================================================
import os, requests
from dotenv import load_dotenv
load_dotenv()

OLLAMA_BASE_URL  = os.environ.get('OLLAMA_BASE_URL',  'http://localhost:11434')
OLLAMA_LLM_MODEL = os.environ.get('OLLAMA_LLM_MODEL', 'llama3.2:3b')

# ---- OPÇÃO A: ChatOllama (RECOMENDADO) ----
def setup_llm_ollama(model: str = OLLAMA_LLM_MODEL, base_url: str = OLLAMA_BASE_URL):
    """Configura LLM via Ollama (cliente nativo langchain-ollama)."""
    from langchain_ollama import ChatOllama
    return ChatOllama(
        model=model,
        base_url=base_url,
        temperature=0.1,        # respostas jurídicas → baixa criatividade
        num_predict=512,        # equivalente a max_tokens
    )

# ---- OPÇÃO B: ChatOpenAI apontando para Ollama /v1 (portabilidade) ----
def setup_llm_ollama_openai_compat(model: str = OLLAMA_LLM_MODEL, base_url: str = OLLAMA_BASE_URL):
    """Mesmo Ollama, mas via interface OpenAI-compatible. Útil para reaproveitar código."""
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(
        model=model,
        base_url=f'{base_url}/v1',
        api_key='ollama',         # Ollama ignora; precisa de qualquer string
        temperature=0.1,
        max_tokens=512,
    )

# ---- OPÇÃO C: HuggingFace Pipeline (sem servidor) — backup offline ----
def setup_llm_huggingface(modelo: str = 'microsoft/phi-2'):
    """Configura LLM via HuggingFace transformers. Para uso sem nenhum servidor."""
    from langchain_community.llms import HuggingFacePipeline
    from transformers import pipeline
    pipe = pipeline(
        'text-generation', model=modelo, max_new_tokens=512,
        temperature=0.1, device_map='auto',
    )
    return HuggingFacePipeline(pipeline=pipe)


# ============================================================
# DETECÇÃO AUTOMÁTICA — tenta Ollama nativo; cai para modo simulado se indisponível
# ============================================================
USE_OPENAI_COMPAT = False   # True força o caminho B (ChatOpenAI → Ollama /v1)

def ollama_ok(base_url: str = OLLAMA_BASE_URL) -> bool:
    try:
        r = requests.get(f'{base_url}/api/tags', timeout=3)
        return r.ok
    except Exception:
        return False


if ollama_ok():
    if USE_OPENAI_COMPAT:
        llm = setup_llm_ollama_openai_compat()
        LLM_NOME = f'ChatOpenAI → Ollama ({OLLAMA_BASE_URL}/v1, modelo={OLLAMA_LLM_MODEL})'
    else:
        llm = setup_llm_ollama()
        LLM_NOME = f'ChatOllama ({OLLAMA_BASE_URL}, modelo={OLLAMA_LLM_MODEL})'
    print(f'✅ Ollama detectado em {OLLAMA_BASE_URL}')
    print(f'   Modelo LLM: {OLLAMA_LLM_MODEL}')
else:
    print(f'⚠️  Ollama não respondeu em {OLLAMA_BASE_URL}.')
    print('   → Suba o servidor da Aula 1 (`ollama serve` ou serviço do SO) e refaça esta célula.')
    print('   → Como fallback de emergência (sem rede), use setup_llm_huggingface() com phi-2.')
    llm = None
    LLM_NOME = 'Não configurado'

print(f'\n🤖 LLM configurado: {LLM_NOME}')


In [ ]:
# ============================================================
# CÉLULA 7b — VERIFICAR / INICIAR O SERVIDOR OLLAMA (da Aula 1)
# ============================================================
# O Ollama foi instalado na Aula 1. Esta célula NÃO o instala —
# apenas verifica se está rodando e puxa os modelos do curso.
#
# Endpoints expostos pelo Ollama (porta 11434):
#   POST http://localhost:11434/api/chat         (API nativa)
#   POST http://localhost:11434/api/embeddings   (API nativa)
#   POST http://localhost:11434/v1/chat/completions  (OpenAI-compatible)
# ============================================================
import subprocess, time, urllib.request, json as _json

OLLAMA_PORT = 11434
OLLAMA_URL  = f'http://localhost:{OLLAMA_PORT}'
MODELOS_NECESSARIOS = ['llama3.2:3b', 'bge-m3']

def ollama_ping():
    try:
        resp = urllib.request.urlopen(f'{OLLAMA_URL}/api/tags', timeout=3)
        return resp.status == 200, _json.loads(resp.read().decode())
    except Exception as e:
        return False, str(e)

ok, payload = ollama_ping()
if not ok:
    print(f'❌ Ollama NÃO está respondendo em {OLLAMA_URL}.')
    print('   Inicie-o conforme o Roteiro da Aula 1:')
    print('     • Windows: abra o app Ollama (ícone de llama na bandeja)')
    print('     • macOS:   `ollama serve &` ou abra o app pelo Launchpad')
    print('     • Linux:   `sudo systemctl start ollama` ou `ollama serve &`')
    print(f'   Detalhe técnico: {payload}')
else:
    instalados = [m['name'] for m in payload.get('models', [])]
    print(f'✅ Ollama OK em {OLLAMA_URL}')
    print(f'   Modelos atualmente instalados: {instalados}')

    faltando = [m for m in MODELOS_NECESSARIOS if m not in instalados]
    if faltando:
        print(f'\n📥 Modelos do curso faltando: {faltando}')
        print('   Execute no terminal (fora deste notebook):')
        for m in faltando:
            print(f'     ollama pull {m}')
        print('\n   (Cada pull leva alguns minutos na primeira vez; depois fica em cache local.)')
    else:
        print('✅ Todos os modelos necessários do curso já estão em cache local.')
        print('➡️  Volte à Célula 7 para configurar o LLM e siga o pipeline.')


## 📝 Célula 8 — Prompt Template Jurídico

O prompt é crucial para qualidade das respostas. Um bom prompt jurídico instrui o LLM a:
- Citar as fontes usadas
- Não inventar informações
- Indicar quando a informação não está no contexto

In [ ]:
# ============================================================
# CÉLULA 8/12 — PROMPT TEMPLATE JURÍDICO
# ============================================================
from langchain.prompts import ChatPromptTemplate

# Prompt calibrado para o contexto jurídico e de segurança pública
# Instruções explícitas para:
#   1. Citar fontes dos documentos recuperados
#   2. Responder apenas com base no contexto
#   3. Admitir quando a informação não está disponível

PROMPT_TEMPLATE = """
Você é um assistente jurídico especializado em Direito Penal e Segurança Pública.
Responda a pergunta com base EXCLUSIVAMENTE nos documentos fornecidos no contexto abaixo.

REGRAS IMPORTANTES:
1. Cite a fonte de cada informação no formato [Fonte: tipo, número]
2. Se a informação não estiver no contexto, responda: "Essa informação não consta nos documentos disponíveis."
3. Não invente dados, jurisprudência ou artigos de lei que não estejam no contexto.
4. Use linguagem técnica jurídica adequada.

CONTEXTO DOS DOCUMENTOS RECUPERADOS:
------------------------------------------------------------
{context}
------------------------------------------------------------

PERGUNTA: {question}

RESPOSTA (com citação das fontes):
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

print("✅ Prompt template configurado")
print()
print("📝 Estrutura do prompt:")
print("   [System instructions para LLM jurídico]")
print("   [Regras anti-alucinação]")
print("   [Context: chunks recuperados pelo FAISS]")
print("   [Question: pergunta do usuário]")

# Verificar o prompt formatado com dados reais
contexto_exemplo = "[Chunk 1 do TJSP aqui...] [Chunk 2 do STJ aqui...]"
pergunta_exemplo = "Quais são os requisitos para prisão preventiva?"
print()
print("📋 Exemplo de prompt formatado (preview):")
print(prompt.format_messages(
    context=contexto_exemplo,
    question=pergunta_exemplo
)[0].content[:400])
print("[...]")

## ⛓️ Célula 9 — Montando a Chain RAG Completa

Conectamos todas as peças: retriever → prompt → LLM → output.

In [ ]:
# ============================================================
# CÉLULA 9/12 — CHAIN RAG COMPLETA (LCEL)
# LCEL = LangChain Expression Language
# ============================================================
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

def formatar_docs(docs):
    """
    Formata os chunks recuperados para inserção no prompt.
    Inclui metadados para facilitar citação.
    """
    partes = []
    for i, doc in enumerate(docs, 1):
        fonte = doc.metadata.get('fonte', 'N/A')
        tipo = doc.metadata.get('tipo', 'N/A')
        numero = doc.metadata.get('numero', 'N/A')
        partes.append(
            f"[DOCUMENTO {i} — Fonte: {fonte} | Tipo: {tipo} | Nº: {numero}]\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n".join(partes)


# Configura o retriever com top-k=3
# k=3: recupera os 3 chunks mais similares à query
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Opções: similarity, mmr, similarity_score_threshold
    search_kwargs={"k": 3}
)

# Monta a chain usando LCEL (pipe operator)
# Fluxo: {context, question} → prompt → llm → string
if llm is not None:
    rag_chain = (
        {
            "context": retriever | formatar_docs,  # Retrieval + formatação
            "question": RunnablePassthrough()       # Passa query direto
        }
        | prompt                                    # Formata o prompt
        | llm                                       # Chama o LLM
        | StrOutputParser()                         # Extrai string da resposta
    )
    print("✅ Chain RAG completa configurada!")
    print(f"   Retriever: FAISS top-3")
    print(f"   LLM: {LLM_NOME}")
    print(f"   Prompt: Template jurídico com anti-alucinação")
else:
    print("⚠️  LLM não configurado. A chain será criada em modo simulado.")
    print("   Configure um LLM na Célula 7 para usar a chain completa.")
    rag_chain = None

print()
print("📐 Diagrama da chain (LCEL):")
print("   retriever | formatar_docs")
print("          +")
print("   RunnablePassthrough (query)")
print("          │")
print("          ▼")
print("   prompt | llm | StrOutputParser")

## 💬 Célula 10 — Rodando o Sistema RAG

Testamos o sistema com perguntas relevantes ao corpus jurídico.

In [ ]:
# ============================================================
# CÉLULA 10/12 — EXECUTANDO O NAIVE RAG
# ============================================================

def perguntar(query: str, mostrar_contexto: bool = False) -> str:
    """
    Faz uma pergunta ao sistema RAG e retorna a resposta.

    Args:
        query: Pergunta do usuário
        mostrar_contexto: Se True, exibe os chunks recuperados

    Returns:
        Resposta gerada pelo LLM
    """
    print(f"\n{'='*65}")
    print(f"❓ PERGUNTA: {query}")
    print('='*65)

    # Recupera contexto para debug/visualização
    docs_recuperados = retriever.invoke(query)

    if mostrar_contexto:
        print("\n📂 Contexto recuperado:")
        for i, doc in enumerate(docs_recuperados, 1):
            print(f"  [{i}] {doc.metadata.get('fonte')} | {doc.metadata.get('tipo')}")
            print(f"      {doc.page_content[:120].strip()!r}")
        print()

    # Gera resposta
    if rag_chain is not None:
        resposta = rag_chain.invoke(query)
    else:
        # Modo simulado (sem LLM)
        contexto = formatar_docs(docs_recuperados)
        resposta = (
            "[MODO SIMULADO — Configure um LLM real para respostas geradas]\n\n"
            f"Com base nos documentos recuperados:\n{contexto[:400]}\n\n"
            "Uma resposta real do LLM apareceria aqui com análise e citações."
        )

    print("\n🤖 RESPOSTA:")
    print(resposta)
    return resposta


# ---- BATERIA DE PERGUNTAS ----

# Pergunta 1: Sobre legislação
r1 = perguntar(
    "Qual é a pena prevista para o crime de tráfico de drogas?",
    mostrar_contexto=True
)

# Pergunta 2: Sobre jurisprudência
r2 = perguntar(
    "Qual o entendimento do STJ sobre prisão preventiva com base em gravidade abstrata do delito?",
    mostrar_contexto=True
)

# Pergunta 3: Sobre inteligência policial
r3 = perguntar(
    "Quantas ocorrências de tráfico de drogas foram registradas no 1º trimestre de 2025?",
    mostrar_contexto=False
)

## 💾 Célula 11 — Persistência e Carregamento do Índice

In [ ]:
# ============================================================
# CÉLULA 11/12 — PERSISTÊNCIA DO ÍNDICE FAISS
# Em produção: salve no Google Drive para reutilizar
# ============================================================

# --- SALVAR ÍNDICE ---
print("💾 Salvando índice FAISS...")
vectorstore.save_local(FAISS_PATH)
print(f"   Índice salvo em: {FAISS_PATH}/")

import os
arquivos = os.listdir(FAISS_PATH)
print(f"   Arquivos: {arquivos}")
for f in arquivos:
    tamanho = os.path.getsize(f"{FAISS_PATH}/{f}")
    print(f"   {f}: {tamanho//1024} KB")

print()

# --- CARREGAR ÍNDICE (em uma nova sessão) ---
print("📂 Carregando índice do disco...")

# allow_dangerous_deserialization=True é necessário para carregar índices FAISS
# É seguro quando você mesmo gerou o índice
vectorstore_carregado = FAISS.load_local(
    FAISS_PATH,
    embeddings_model,
    allow_dangerous_deserialization=True  # Necessário desde LangChain v0.1.6
)

print(f"✅ Índice carregado: {vectorstore_carregado.index.ntotal} vetores")

# Verifica que o índice carregado funciona
teste = vectorstore_carregado.similarity_search("tráfico de drogas", k=1)
print(f"   Teste de busca OK: encontrou doc de '{teste[0].metadata.get('tipo')}'")

print()
print("💡 Para salvar no Google Drive:")
print("   from google.colab import drive")
print("   drive.mount('/content/drive')")
print("   vectorstore.save_local('/content/drive/MyDrive/faiss_juridico')")

---
## ⚠️ Célula 12 — Armadilhas Comuns (Top 5)

Esta seção documenta os **5 erros mais frequentes** neste lab, com diagnóstico e solução.

In [ ]:
# ============================================================
# CÉLULA 12/12 — ARMADILHAS COMUNS: DEMONSTRAÇÃO E DIAGNÓSTICO
# ============================================================

print("⚠️  ARMADILHAS COMUNS NO NAIVE RAG — TOP 5")
print("="*65)
print()

# ============================================================
# ARMADILHA 1: Chunk Size Muito Pequeno
# ============================================================
print("❌ ARMADILHA 1: Chunk Size Muito Pequeno")
print("-"*50)

splitter_errado = RecursiveCharacterTextSplitter(
    chunk_size=100,   # ← MUITO PEQUENO para textos jurídicos
    chunk_overlap=10,
)
chunks_errados = splitter_errado.split_documents([CORPUS_JURIDICO[0]])

print(f"Chunks gerados com size=100: {len(chunks_errados)}")
print(f"Primeiro chunk: {chunks_errados[0].page_content!r}")
print()
print("🔴 Problema: contexto insuficiente → resposta incompleta")
print("✅ Solução: usar mínimo 500-800 chars para textos jurídicos")
print()

# ============================================================
# ARMADILHA 2: Sem Overlap → Perde Informação nas Fronteiras
# ============================================================
print("❌ ARMADILHA 2: Overlap Zero")
print("-"*50)

# Simula texto onde informação crucial está "na fronteira"
texto_borda = "...O réu foi condenado por tráfico de drogas em\nPrimeira instância. A pena foi fixada em..."
s_sem_overlap = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
s_com_overlap = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=20)

c_sem = s_sem_overlap.split_text(texto_borda)
c_com = s_com_overlap.split_text(texto_borda)

print(f"Sem overlap — Chunks: {len(c_sem)}")
for i, c in enumerate(c_sem):
    print(f"  [{i}] {c.strip()!r}")
print(f"Com overlap — Chunks: {len(c_com)}")
for i, c in enumerate(c_com):
    print(f"  [{i}] {c.strip()!r}")

print()
print("🔴 Problema sem overlap: 'em Primeira instância' cortado no meio")
print("✅ Solução: usar chunk_overlap=10-20% do chunk_size")
print()

# ============================================================
# ARMADILHA 3: Sem Metadados de Fonte
# ============================================================
print("❌ ARMADILHA 3: Documents sem Metadados")
print("-"*50)

# Errado: Document sem metadados
doc_sem_meta = Document(page_content="A prisão preventiva é medida excepcional...")
print(f"SEM metadados: {doc_sem_meta.metadata}")
print("🔴 Problema: impossível citar fonte → resposta sem embasamento")

# Correto: Document com metadados
doc_com_meta = Document(
    page_content="A prisão preventiva é medida excepcional...",
    metadata={"fonte": "TJSP", "tipo": "acórdão", "numero": "HC-2025.001234"}
)
print(f"COM metadados: {doc_com_meta.metadata}")
print("✅ Solução: sempre incluir source, tipo, número, data")
print()

# ============================================================
# ARMADILHA 4: Modelo de Embedding Inconsistente
# ============================================================
print("❌ ARMADILHA 4: Modelos de Embedding Diferentes")
print("-"*50)
print("Indexação: modelo A → vetores no espaço X")
print("Query:     modelo B → vetor no espaço Y")
print("Resultado: busca retorna documentos aleatórios (não relacionados)")
print()

# Demonstração do problema
v_modelo_a = embeddings_model.embed_query("prisão preventiva")
print(f"Vetor modelo A (dim {len(v_modelo_a)}): {v_modelo_a[:3]}...")
print("Se usarmos outro modelo: vetores com dimensão ou escala diferente")
print("🔴 Problema: resultados de busca completamente errados")
print("✅ Solução: variável EMBEDDING_MODEL define o modelo UMA VEZ")
print(f"            Modelo atual: {EMBEDDING_MODEL}")
print()

# ============================================================
# ARMADILHA 5: PyPDF2 em PDFs Escaneados
# ============================================================
print("❌ ARMADILHA 5: PyPDF2 em PDFs Escaneados")
print("-"*50)
print("PyPDF2.extract_text() retorna string vazia ou:\n")
print("  texto_extraido = 'ÿþ\\x00M\\x00i\\x00n....' (lixo de binário)")
print()
print("🔴 Problema: sem texto → sem chunks → sem retrieval → sem resposta")
print("✅ Solução: usar Docling com do_ocr=True para PDFs escaneados")
print()
print("Código de diagnóstico:")
print("""
def diagnosticar_pdf(caminho):
    import PyPDF2
    with open(caminho, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        texto = reader.pages[0].extract_text()
        if not texto or len(texto) < 50:
            print("⚠️  PDF provavelmente escaneado — use Docling com OCR")
            return False
        print(f"✅ PDF com texto nativo ({len(texto)} chars)")
        return True
""")

print()
print("="*65)
print("📚 RESUMO DAS ARMADILHAS:")
print("  1. chunk_size muito pequeno → use ≥ 500 chars")
print("  2. overlap=0 → use 10-20% do chunk_size")
print("  3. sem metadados → sempre adicione source/tipo/número")
print("  4. modelos de embedding diferentes → defina EMBEDDING_MODEL")
print("  5. PyPDF2 em PDFs escaneados → use Docling com OCR")

---
## 🏋️ Exercício de Extensão — Adicionar Novo Documento ao Índice

**Objetivo:** Adicionar um novo documento ao índice existente **sem re-indexar tudo**.

Esta é uma habilidade essencial em produção, onde novos documentos chegam continuamente.

In [ ]:
# ============================================================
# EXERCÍCIO — ADICIONAR NOVO DOCUMENTO AO ÍNDICE
# ============================================================
print("🏋️  EXERCÍCIO: Adicionando novo documento ao índice existente")
print("="*65)
print(f"Índice atual: {vectorstore.index.ntotal} vetores")
print()

# ---- NOVO DOCUMENTO: Decreto municipal sobre segurança ----
NOVO_DOCUMENTO = Document(
    page_content="""
    DECRETO MUNICIPAL Nº 12.345/2025
    Prefeitura do Município de São Paulo

    Dispõe sobre a criação do Programa Municipal de Prevenção à Criminalidade (PMPC)
    e estabelece diretrizes para a integração das forças de segurança no município.

    Art. 1º Fica criado o Programa Municipal de Prevenção à Criminalidade (PMPC),
    com os seguintes objetivos:
    I - reduzir os índices de criminalidade em 15% no prazo de 24 meses;
    II - promover a integração entre Guarda Civil Metropolitana, Polícia Militar e Polícia Civil;
    III - implementar câmeras de monitoramento em 50 pontos críticos identificados pelo DENARC.

    Art. 2º O PMPC contará com dotação orçamentária de R$ 50.000.000,00 (cinquenta milhões)
    para o exercício de 2025, conforme Lei Orçamentária Municipal nº 9.876/2024.

    Art. 3º Este decreto entra em vigor na data de sua publicação.
    """,
    metadata={
        "fonte": "Prefeitura de São Paulo",
        "tipo": "decreto_municipal",
        "numero": "12.345/2025",
        "assunto": "segurança pública - programa municipal - prevenção",
        "data": "2025-04-01"
    }
)

# PASSO 1: Chunkizar o novo documento
novos_chunks = text_splitter.split_documents([NOVO_DOCUMENTO])
print(f"Novo documento chunkizado: {len(novos_chunks)} chunks")

# PASSO 2: Adicionar ao índice existente (sem re-indexar tudo)
# add_documents() é muito mais eficiente que recriar o índice do zero
ids_novos = vectorstore.add_documents(novos_chunks)

print(f"✅ Documentos adicionados! IDs gerados: {ids_novos}")
print(f"Índice após adição: {vectorstore.index.ntotal} vetores")

# PASSO 3: Verificar que o novo documento é recuperável
print()
print("🔍 Testando busca no novo documento:")
resultados = vectorstore.similarity_search(
    "programa municipal de prevenção à criminalidade",
    k=2
)
for i, doc in enumerate(resultados, 1):
    print(f"  [{i}] Tipo: {doc.metadata.get('tipo')} | Fonte: {doc.metadata.get('fonte')}")
    print(f"       {doc.page_content[:150].strip()!r}")

# PASSO 4: Salvar índice atualizado
vectorstore.save_local(FAISS_PATH)
print()
print(f"✅ Índice atualizado salvo em: {FAISS_PATH}")

print()
print("━"*65)
print("📝 TAREFA DO ALUNO:")
print("   Adicione 2 novos documentos ao índice:")
print("   1. Uma notícia recente sobre segurança pública")
print("   2. Um artigo acadêmico sobre criminologia")
print("   Verifique que ambos são recuperados por queries relevantes.")

---
## ✅ Conclusão do LAB 1

Você construiu um **Naive RAG completo** do zero. O sistema pode:
- Ingerir e chunkizar documentos jurídicos
- Gerar embeddings e indexar no FAISS
- Recuperar chunks relevantes por similaridade semântica
- Gerar respostas fundamentadas com citação de fontes
- Persistir e carregar o índice
- Adicionar novos documentos sem re-indexar

**Próximos laboratórios:**
- LAB 2: Avaliando o RAG com métricas (Ragas, BLEU, ROUGE)
- Aula 3: RAG Modular — reranking, compressão, query expansion

> **ABNT:** LANGCHAIN. *LCEL (LangChain Expression Language)*. Docs, 2024. Disponível em: <https://python.langchain.com/docs/expression_language>.